# Interactive Topology Optimization

Pick a problem type and its size/volume-fraction, pick a model and its
hyperparameters, then run the cell below the controls. A progress line is
printed every 25 steps while the optimization runs, and the final run
produces:

- **Compliance** (final/best objective value)
- **Gray fraction** (how far the design is from a crisp 0/1 material layout)
- **Topology PNG** (saved to `interactive_outputs/` and shown inline)
- **Elapsed time** for the optimization run

**Instructions**
1. Run the *Setup* cell once per kernel session.
2. Run the *Controls* cell to display the widgets.
3. Adjust the problem/model controls, then click **Run Optimization**.
4. Re-click the button any time after changing a control to re-run.

## Problems

Problem choices are the exact preconfigured cases already defined in
`neural_structural_optimization.problems.PROBLEMS_BY_CATEGORY` (the same set
listed in `problems.txt`) — pick a **Category** (e.g. `mbb_beam`, `l_shape_0.4`,
`thin_support_bridge`) and then a specific **Config** (its fixed
width x height and target density). There are no free-form size/density
sliders: each config's dimensions are already known to work with every model
below.

## Models

- **Pixel - LBFGS / MMA / OC**: direct per-pixel density, optimized with
  L-BFGS, the Method of Moving Asymptotes (`nlopt`), or Optimality Criteria.
  No extra hyperparameters.
- **CNN - LBFGS**: a convolutional generator (`CNNModel`) reparameterizes the
  density field from a latent vector. Configure `latent_size` and
  `dense_channels`.
- **Hybrid KAN - LBFGS**: `HybridKANModel` — same CNN decoder as above, but a
  KAN learns per-channel gating from the latent vector. Configure
  `latent_size`, `hidden_size`, `num_kan_layers`, `grid`, `spline order (k)`.
- **Base KAN - LBFGS**: `BaseKANModel` — a coordinate KAN mapping `(x, y)`
  directly to density. Configure hidden layer widths, `grid`, `spline order (k)`.

Note: CNN and Hybrid KAN use a fixed 4x spatial upsampling internally, so
`width` and `height` must both be divisible by 4 — true for every config in
`problems.py`, so this is handled automatically.

In [9]:
import sys

try:
    import ipywidgets  # noqa: F401
except ImportError:
    get_ipython().system(f"{sys.executable} -m pip install -q ipywidgets")

# Force the inline backend so figures always render as output in this
# notebook instead of popping up in a separate GUI window.
get_ipython().run_line_magic("matplotlib", "inline")

# Auto-reload local modules (models.py, problems.py, ...) on every cell run so
# edits to those files take effect without restarting the kernel.
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

import os
import threading
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.ioff()  # never auto-open a GUI window; figures are shown via display()

from neural_structural_optimization import problems, topo_api
from models import (
    PixelModel,
    CNNModel,
    HybridKANModel,
    BaseKANModel,
    train_lbfgs,
    method_of_moving_asymptotes,
    optimality_criteria,
)

OUTPUT_DIR = "interactive_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROGRESS_EVERY = 25  # print a loss update every this many optimization steps

# Fixed CNN / Hybrid-KAN spatial decoder shape: 4x total upsampling so that
# any width/height divisible by 4 works out of the box.
DECODER_RESIZES = (1, 2, 2, 1)
DECODER_CONV_FILTERS = (64, 32, 16, 1)

CATEGORIES = sorted(problems.PROBLEMS_BY_CATEGORY)

MODEL_OPTIONS = [
    "Pixel - LBFGS",
    "Pixel - MMA",
    "Pixel - OC",
    "CNN - LBFGS",
    "Hybrid KAN - LBFGS",
    "Base KAN - LBFGS",
]

print("Setup complete: {} problem categories ({} configs), {} models available.".format(
    len(CATEGORIES), len(problems.PROBLEMS_BY_NAME), len(MODEL_OPTIONS)))


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setup complete: 28 problem categories (116 configs), 6 models available.


In [10]:
# --- Interactive Controls ---
style = {"description_width": "initial"}

def _config_options(category):
    """(label, name) pairs for every preset config in this category,
    e.g. label='96 x 32, density=0.5' -> name='mbb_beam_96x32_0.5'."""
    return [
        ("{} x {}, density={}".format(p.width, p.height, p.density), p.name)
        for p in problems.PROBLEMS_BY_CATEGORY[category]
    ]


problem_category = widgets.Dropdown(
    options=CATEGORIES,
    value=CATEGORIES[0],
    description="Category:",
    style=style,
)
problem_config = widgets.Dropdown(
    options=_config_options(problem_category.value),
    description="Config:",
    style=style,
)


def on_category_change(change):
    problem_config.options = _config_options(change["new"])


problem_category.observe(on_category_change, names="value")

model_name = widgets.Dropdown(options=MODEL_OPTIONS, value="Pixel - LBFGS", description="Model:", style=style)

# Max step count: a slider for quick adjustment, linked to a type-in box so
# an exact value can be entered directly.
max_iterations = widgets.IntSlider(value=100, min=10, max=2000, step=10, description="Max steps:", style=style)
max_iterations_text = widgets.BoundedIntText(value=100, min=1, max=100000, description="(exact):", style=style, layout=widgets.Layout(width="160px"))
widgets.jslink((max_iterations, "value"), (max_iterations_text, "value"))
max_steps_box = widgets.HBox([max_iterations, max_iterations_text])

# CNN params
cnn_latent_size = widgets.IntSlider(value=128, min=16, max=256, step=16, description="Latent size:", style=style)
cnn_dense_channels = widgets.IntSlider(value=32, min=8, max=64, step=8, description="Dense channels:", style=style)
cnn_box = widgets.VBox([widgets.HTML("<b>CNN parameters</b>"), cnn_latent_size, cnn_dense_channels])

# Hybrid KAN params
hy_latent_size = widgets.IntSlider(value=128, min=16, max=256, step=16, description="Latent size:", style=style)
hy_hidden_size = widgets.IntSlider(value=64, min=8, max=128, step=8, description="Hidden size:", style=style)
hy_num_kan_layers = widgets.IntSlider(value=1, min=1, max=3, step=1, description="KAN layers:", style=style)
hy_grid = widgets.IntSlider(value=5, min=2, max=20, step=1, description="Grid size:", style=style)
hy_k = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Spline order (k):", style=style)
hybrid_box = widgets.VBox([
    widgets.HTML("<b>Hybrid KAN parameters</b>"),
    hy_latent_size, hy_hidden_size, hy_num_kan_layers, hy_grid, hy_k,
])

# Base KAN params
kan_layers_text = widgets.Text(value="16, 16", description="Hidden layers:", style=style)
kan_grid = widgets.IntSlider(value=8, min=2, max=100, step=1, description="Grid size:", style=style)
kan_k = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Spline order (k):", style=style)
basekan_box = widgets.VBox([
    widgets.HTML("<b>Base KAN parameters</b>"),
    kan_layers_text, kan_grid, kan_k,
])

param_boxes = {
    "CNN - LBFGS": cnn_box,
    "Hybrid KAN - LBFGS": hybrid_box,
    "Base KAN - LBFGS": basekan_box,
}


def on_model_change(change):
    selected = change["new"]
    for name, box in param_boxes.items():
        box.layout.display = "flex" if name == selected else "none"


model_name.observe(on_model_change, names="value")

run_button = widgets.Button(description="Run Optimization", button_style="success", icon="play")
output = widgets.Output()

# Some notebook front-ends (observed in VS Code's Jupyter extension) can
# dispatch a single button click as two overlapping callback invocations,
# which interleave their prints. Guard with a non-blocking lock so any
# duplicate/overlapping invocation is dropped immediately instead of running
# concurrently with the first.
_run_lock = threading.Lock()


def run_optimization(_button):
    if not _run_lock.acquire(blocking=False):
        return  # a run is already in progress (likely a duplicate click event)
    run_button.disabled = True
    try:
      with output:
        clear_output(wait=True)
        try:
            p_name = problem_config.value
            problem = problems.PROBLEMS_BY_NAME[p_name]
            args = topo_api.specified_task(problem)

            m_name = model_name.value
            iters = max_iterations.value

            print("Problem: {} ({})".format(p_name, problem_category.value))
            print("Domain: {} x {}, target density: {}".format(problem.width, problem.height, problem.density))
            print("Model: {}, Max steps: {}".format(m_name, iters))

            needs_decoder = m_name in ("CNN - LBFGS", "Hybrid KAN - LBFGS")
            total_resize = int(np.prod(DECODER_RESIZES))
            if needs_decoder and (problem.width % total_resize or problem.height % total_resize):
                raise ValueError(
                    "{} requires width and height to both be divisible by {} "
                    "(got {}x{}). Pick a different config.".format(
                        m_name, total_resize, problem.width, problem.height)
                )

            print("Optimizing...")
            t0 = time.time()
            if m_name == "Pixel - LBFGS":
                model = PixelModel(seed=0, args=args)
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Pixel - MMA":
                model = PixelModel(seed=0, args=args)
                ds = method_of_moving_asymptotes(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Pixel - OC":
                model = PixelModel(seed=0, args=args)
                ds = optimality_criteria(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "CNN - LBFGS":
                model = CNNModel(
                    seed=0, args=args,
                    latent_size=cnn_latent_size.value,
                    dense_channels=cnn_dense_channels.value,
                    resizes=DECODER_RESIZES,
                    conv_filters=DECODER_CONV_FILTERS,
                )
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Hybrid KAN - LBFGS":
                model = HybridKANModel(
                    seed=0, args=args,
                    latent_size=hy_latent_size.value,
                    hidden_size=hy_hidden_size.value,
                    num_kan_layers=hy_num_kan_layers.value,
                    grid=hy_grid.value,
                    k=hy_k.value,
                    resizes=DECODER_RESIZES,
                    conv_filters=DECODER_CONV_FILTERS,
                )
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            elif m_name == "Base KAN - LBFGS":
                layers = tuple(int(x.strip()) for x in kan_layers_text.value.split(",") if x.strip())
                model = BaseKANModel(
                    seed=0, args=args,
                    kan_layers=layers,
                    grid=kan_grid.value,
                    k=kan_k.value,
                )
                ds = train_lbfgs(model, iters, progress_every=PROGRESS_EVERY)
            else:
                raise ValueError("Unknown model option: {}".format(m_name))
            elapsed = time.time() - t0

            losses = ds.loss.values
            valid_mask = ~np.isnan(losses)
            best_step = int(np.nanargmin(losses))
            compliance = float(losses[valid_mask].min())

            final_design = np.clip(ds.design.isel(step=best_step).values, 0.0, 1.0)
            gray_fraction = float(np.mean(4.0 * final_design * (1.0 - final_design)))

            safe_model = m_name.replace(" ", "_").replace("/", "_")
            png_path = os.path.join(OUTPUT_DIR, "{}_{}.png".format(p_name, safe_model))
            plt.imsave(png_path, 1.0 - final_design, cmap="gray", vmin=0.0, vmax=1.0)

            print()
            print("=" * 55)
            print("Compliance (best loss):  {:.4f}  (step {})".format(compliance, best_step))
            print("Gray fraction:           {:.4f}".format(gray_fraction))
            print("Elapsed time:            {:.2f} s".format(elapsed))
            print("Topology PNG saved to:   {}".format(png_path))
            print("=" * 55)

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

            ax1.imshow(1.0 - final_design, cmap="gray", vmin=0.0, vmax=1.0)
            ax1.set_title("Final design: {} / {}".format(p_name, m_name))
            ax1.axis("off")

            ax2.plot(np.arange(len(losses)), losses)
            ax2.axvline(best_step, color="red", linestyle="--", linewidth=1, label="best step")
            ax2.set_title("Compliance vs. iteration")
            ax2.set_xlabel("Iteration")
            ax2.set_ylabel("Compliance")
            ax2.legend()
            ax2.grid(True)

            plt.tight_layout()
            display(fig)   # render inline in the notebook output, never a popup window
            plt.close(fig)
        except Exception:
            traceback.print_exc()
    finally:
        run_button.disabled = False
        _run_lock.release()


run_button.on_click(run_optimization)

# Initial layout
on_model_change({"new": model_name.value})

display(widgets.VBox([
    widgets.HTML("<h3>Problem parameters</h3>"),
    problem_category, problem_config,
    widgets.HTML("<hr><h3>Model parameters</h3>"),
    model_name, max_steps_box,
    cnn_box, hybrid_box, basekan_box,
    run_button,
]))
display(output)


Output()

---

# KAN Reusability: Cross-Grid / Cross-Problem Transfer

`BaseKANModel` maps *normalized* coordinates `(x, y) \in [-1, 1]^2` to a density
value, independent of how many grid cells (`nelx`, `nely`) that domain is
divided into (see `models.py` — `cx`/`cy` are always rescaled to `[-1, 1]`
before being fed to the KAN). That means the learned KAN body
(`model.kan`, the actual B-spline network) is a **resolution-independent
function of position** and can, in principle, be copied from a model
trained on a small grid directly into a model built for a larger grid, or
even a differently-shaped domain / different problem type — something a
pixel or CNN-based parameterization cannot do, since CNN feature maps and
per-pixel densities are tied to a fixed spatial resolution.

The actual payoff we care about is **training time**: full topology
optimization runs (especially at larger grids) can take a long time, so the
interesting question is not just "does transfer help at all" but **how much
of that training time can be skipped** by reusing a KAN pre-trained on a
small, cheap problem. This section trains a `BaseKANModel` on a small
**source** problem, then transfers only its `.kan` submodule weights into a
fresh model built for a **target** problem (a bigger grid of the *same*
problem type, or a completely different problem type). It then compares:

1. **From scratch (full budget)** — target model trained from a random init
   for `Scratch steps (full budget)` steps. This is the "no reuse" baseline
   and its final compliance is treated as the quality bar to beat.
2. **Zero-shot transfer** — source KAN weights copied in, **no** further
   training on the target problem at all (0 seconds of target-problem
   training).
3. **Fine-tuned transfer** — source KAN weights copied in, then trained on
   the target problem, checking after every step whether it has already
   reached the from-scratch bar.

The headline numbers this section reports are: **how many fine-tuning steps**
(and **how many seconds**) it took the transferred KAN to match or beat the
fully-trained-from-scratch compliance — i.e. the concrete training time saved
by reusing a KAN instead of training the target problem from scratch. If the
transferred model never catches up within its allotted budget, that is
reported too (KAN reuse isn't free — a source problem too different from the
target can transfer poorly).

In [ ]:
# --- KAN Reusability Controls ---
import copy

import torch

transfer_style = {"description_width": "initial"}

source_category = widgets.Dropdown(
    options=CATEGORIES, value=CATEGORIES[0], description="Source category:", style=transfer_style,
)
source_config = widgets.Dropdown(
    options=_config_options(source_category.value), description="Source config:", style=transfer_style,
)
target_category = widgets.Dropdown(
    options=CATEGORIES, value=CATEGORIES[0], description="Target category:", style=transfer_style,
)
target_config = widgets.Dropdown(
    options=_config_options(target_category.value), description="Target config:", style=transfer_style,
)


def _on_source_category_change(change):
    source_config.options = _config_options(change["new"])


def _on_target_category_change(change):
    target_config.options = _config_options(change["new"])


source_category.observe(_on_source_category_change, names="value")
target_category.observe(_on_target_category_change, names="value")

# Pre-select a natural small -> big example: mbb_beam 96x32 -> 192x64 if
# available, so the section works out of the box before any clicks.
if "mbb_beam_96x32_0.5" in problems.PROBLEMS_BY_NAME:
    source_category.value = "mbb_beam"
    source_config.options = _config_options("mbb_beam")
    source_config.value = "mbb_beam_96x32_0.5"
if "mbb_beam_192x64_0.4" in problems.PROBLEMS_BY_NAME:
    target_category.value = "mbb_beam"
    target_config.options = _config_options("mbb_beam")
    target_config.value = "mbb_beam_192x64_0.4"

transfer_kan_layers_text = widgets.Text(value="16, 16", description="Hidden layers:", style=transfer_style)
transfer_grid = widgets.IntSlider(value=8, min=2, max=100, step=1, description="Grid size:", style=transfer_style)
transfer_k = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Spline order (k):", style=transfer_style)

pretrain_steps = widgets.BoundedIntText(value=200, min=0, max=100000, description="Pre-train steps (source):", style=transfer_style)
scratch_steps = widgets.BoundedIntText(value=300, min=1, max=100000, description="Scratch steps (full budget):", style=transfer_style)
finetune_budget = widgets.BoundedIntText(value=300, min=1, max=100000, description="Fine-tune budget (target, transfer):", style=transfer_style)

transfer_run_button = widgets.Button(description="Run Transfer Comparison", button_style="success", icon="play")
transfer_output = widgets.Output()

_transfer_run_lock = threading.Lock()


def _build_base_kan(args, seed):
    layers = tuple(int(x.strip()) for x in transfer_kan_layers_text.value.split(",") if x.strip())
    return BaseKANModel(seed=seed, args=args, kan_layers=layers, grid=transfer_grid.value, k=transfer_k.value)


def _best_compliance_and_design(ds):
    losses = ds.loss.values
    valid_mask = ~np.isnan(losses)
    best_step = int(np.nanargmin(losses))
    compliance = float(losses[valid_mask].min())
    design = np.clip(ds.design.isel(step=best_step).values, 0.0, 1.0)
    return compliance, design, losses


def _approx_step_times(total_time, n_steps):
    """train_lbfgs hands its whole loop to scipy's L-BFGS-B in one call, so no
    true per-step timestamp is available. Approximate one by spreading the
    measured wall-clock time uniformly over the recorded steps -- L-BFGS-B
    function evaluations on a fixed-size model take roughly constant time, so
    this is a reasonable stand-in for "how far into training, time-wise, was
    step i" without instrumenting scipy's C loop."""
    return np.linspace(total_time / n_steps, total_time, n_steps)


def _first_step_reaching(losses, times, target_compliance):
    """Return (step, wall_clock_seconds) of the first step whose compliance
    is <= target_compliance, or (None, None) if it never gets there."""
    hits = np.flatnonzero(losses <= target_compliance)
    if hits.size == 0:
        return None, None
    step = int(hits[0])
    return step, float(times[step])


def run_transfer_comparison(_button):
    if not _transfer_run_lock.acquire(blocking=False):
        return
    transfer_run_button.disabled = True
    try:
      with transfer_output:
        clear_output(wait=True)
        try:
            src_name = source_config.value
            tgt_name = target_config.value
            src_problem = problems.PROBLEMS_BY_NAME[src_name]
            tgt_problem = problems.PROBLEMS_BY_NAME[tgt_name]
            src_args = topo_api.specified_task(src_problem)
            tgt_args = topo_api.specified_task(tgt_problem)

            print("Source: {} ({} x {}, density={})".format(
                src_name, src_problem.width, src_problem.height, src_problem.density))
            print("Target: {} ({} x {}, density={})".format(
                tgt_name, tgt_problem.width, tgt_problem.height, tgt_problem.density))
            print()

            # 1. Pre-train the source KAN (this cost is already "sunk" --
            #    e.g. it may have been done once, earlier, on a cheap small
            #    grid -- so it is reported separately from the target-side
            #    training time comparison below).
            print("[1/4] Pre-training source KAN for {} steps (sunk cost, done once)...".format(pretrain_steps.value))
            t0 = time.time()
            src_model = _build_base_kan(src_args, seed=0)
            if pretrain_steps.value > 0:
                train_lbfgs(src_model, pretrain_steps.value, progress_every=PROGRESS_EVERY,
                            save_intermediate_designs=False)
            pretrain_time = time.time() - t0
            print("  done in {:.2f}s".format(pretrain_time))

            # 2. From-scratch baseline on the target problem, full budget,
            #    same optimizer (L-BFGS) the fine-tuned run will use below.
            #    This is the quality bar: what training this problem "for
            #    real" (no reuse) actually costs and achieves.
            print("[2/4] Training target KAN from scratch for {} steps (full budget, no reuse)...".format(scratch_steps.value))
            t0 = time.time()
            scratch_model = _build_base_kan(tgt_args, seed=1)
            ds_scratch = train_lbfgs(scratch_model, scratch_steps.value, progress_every=PROGRESS_EVERY)
            scratch_compliance, scratch_design, scratch_losses = _best_compliance_and_design(ds_scratch)
            scratch_time = time.time() - t0
            print("  done in {:.2f}s, best compliance={:.4f}".format(scratch_time, scratch_compliance))
            scratch_times_approx = _approx_step_times(scratch_time, len(scratch_losses))

            # 3. Zero-shot transfer: copy the source KAN weights, no training
            #    on the target problem at all -- 0 seconds of target-side cost.
            print("[3/4] Evaluating zero-shot transfer (0s of target-problem training)...")
            zeroshot_model = _build_base_kan(tgt_args, seed=1)
            zeroshot_model.kan.load_state_dict(copy.deepcopy(src_model.kan.state_dict()))
            with torch.no_grad():
                logits = zeroshot_model()
                zeroshot_compliance = float(zeroshot_model.loss(logits).item())
                zeroshot_design = zeroshot_model.env.render(
                    logits.detach().cpu().numpy().reshape(-1), volume_contraint=True)
            print("  zero-shot compliance={:.4f}".format(zeroshot_compliance))

            # 4. Fine-tuned transfer: copy the source KAN weights, then
            #    resume training with the *same* optimizer (L-BFGS) as the
            #    scratch run, so the two curves are optimizer-matched and any
            #    speedup is attributable to the transferred weights alone.
            print("[4/4] Fine-tuning transferred KAN for up to {} steps, checking for crossover...".format(finetune_budget.value))
            t0 = time.time()
            finetune_model = _build_base_kan(tgt_args, seed=1)
            finetune_model.kan.load_state_dict(copy.deepcopy(src_model.kan.state_dict()))
            ds_finetune = train_lbfgs(finetune_model, finetune_budget.value, progress_every=PROGRESS_EVERY)
            finetune_compliance, finetune_design, finetune_losses = _best_compliance_and_design(ds_finetune)
            finetune_time = time.time() - t0
            print("  done in {:.2f}s, best compliance={:.4f}".format(finetune_time, finetune_compliance))
            finetune_times_approx = _approx_step_times(finetune_time, len(finetune_losses))

            crossover_step, crossover_time = _first_step_reaching(
                finetune_losses, finetune_times_approx, scratch_compliance)

            print()
            print("=" * 70)
            print("{:<30} {:>15} {:>20}".format("Condition", "Compliance", "Target-side time"))
            print("-" * 70)
            print("{:<30} {:>15.4f} {:>18.2f}s".format("From scratch (full budget)", scratch_compliance, scratch_time))
            print("{:<30} {:>15.4f} {:>18.2f}s".format("Zero-shot transfer", zeroshot_compliance, 0.0))
            print("{:<30} {:>15.4f} {:>18.2f}s".format("Fine-tuned transfer (best)", finetune_compliance, finetune_time))
            print("=" * 70)
            print()
            print("(Source KAN pre-training took {:.2f}s -- a one-time, sunk cost;".format(pretrain_time))
            print(" it is not counted against the target-side times above.)")
            print()

            if crossover_step is not None:
                time_saved = scratch_time - crossover_time
                pct_saved = 100.0 * time_saved / scratch_time if scratch_time > 0 else float("nan")
                print("RESULT: fine-tuned transfer matched the from-scratch compliance")
                print("        ({:.4f}) after only {} step(s) / ~{:.2f}s of target-problem".format(
                    scratch_compliance, crossover_step + 1, crossover_time))
                print("        training, vs. {:.2f}s to train from scratch.".format(scratch_time))
                print("        --> estimated training time saved: ~{:.2f}s ({:.1f}%)".format(time_saved, pct_saved))
                print("        (step times are approximate -- see _approx_step_times)")
            else:
                print("RESULT: fine-tuned transfer did NOT reach the from-scratch compliance")
                print("        ({:.4f}) within its {}-step budget (best reached: {:.4f}).".format(
                    scratch_compliance, finetune_budget.value, finetune_compliance))
                print("        Try a larger 'Fine-tune budget', or a source problem more")
                print("        similar to the target -- KAN reuse is not guaranteed to help")
                print("        if the two problems' optimal density fields are too different.")

            fig, axes = plt.subplots(2, 2, figsize=(11, 9))
            ((ax_scratch, ax_zero), (ax_fine, ax_curve)) = axes

            ax_scratch.imshow(1.0 - scratch_design, cmap="gray", vmin=0.0, vmax=1.0)
            ax_scratch.set_title("From scratch (full budget)\ncompliance={:.4f}, {:.1f}s".format(scratch_compliance, scratch_time))
            ax_scratch.axis("off")

            ax_zero.imshow(1.0 - zeroshot_design, cmap="gray", vmin=0.0, vmax=1.0)
            ax_zero.set_title("Zero-shot transfer\ncompliance={:.4f}, 0.0s".format(zeroshot_compliance))
            ax_zero.axis("off")

            ax_fine.imshow(1.0 - finetune_design, cmap="gray", vmin=0.0, vmax=1.0)
            ax_fine.set_title("Fine-tuned transfer (best)\ncompliance={:.4f}, {:.1f}s".format(finetune_compliance, finetune_time))
            ax_fine.axis("off")

            ax_curve.plot(scratch_times_approx, scratch_losses, label="from scratch")
            ax_curve.plot(finetune_times_approx, finetune_losses, label="fine-tuned transfer")
            ax_curve.axhline(scratch_compliance, color="black", linestyle=":", linewidth=1, label="from-scratch final compliance")
            ax_curve.axhline(zeroshot_compliance, color="gray", linestyle="--", linewidth=1, label="zero-shot (no training)")
            if crossover_time is not None:
                ax_curve.axvline(crossover_time, color="red", linestyle="--", linewidth=1,
                                  label="transfer crossover (~{:.1f}s)".format(crossover_time))
            ax_curve.set_title("Compliance vs. target-problem training time (both use L-BFGS)")
            ax_curve.set_xlabel("Target-problem training time (s, approximate)")
            ax_curve.set_ylabel("Compliance")
            ax_curve.legend(fontsize=8)
            ax_curve.grid(True)

            plt.tight_layout()
            display(fig)
            plt.close(fig)
        except Exception:
            traceback.print_exc()
    finally:
        transfer_run_button.disabled = False
        _transfer_run_lock.release()


transfer_run_button.on_click(run_transfer_comparison)

display(widgets.VBox([
    widgets.HTML("<h3>Source problem (small, used to pre-train the KAN)</h3>"),
    source_category, source_config,
    widgets.HTML("<h3>Target problem (transfer destination -- bigger grid and/or different type)</h3>"),
    target_category, target_config,
    widgets.HTML("<hr><h3>Shared Base KAN architecture</h3>"),
    transfer_kan_layers_text, transfer_grid, transfer_k,
    widgets.HTML("<hr><h3>Training budget</h3>"),
    pretrain_steps, scratch_steps, finetune_budget,
    transfer_run_button,
]))
display(transfer_output)

Output()

---

# KAN Grid Refinement: train coarse, expand the spline grid mid-run

This tests the KAN-unique coarse-to-fine strategy (`BaseKANModel.refine()` +
`train_lbfgs_adaptive_kan`): start with a coarse B-spline grid (cheap, smooth,
finds the rough load path), then expand the grid mid-run — the learned
functions are least-squares refitted onto the finer basis so nothing is lost —
and continue training to add fine structural detail. The analogue of
h-refinement in adaptive FEM; CNN/pixel models have no equivalent operation.

**The test**: same problem, same seed, same total step budget, two runs —

1. **Adaptive** — follows your schedule of `steps:grid` phases
   (e.g. `100:4, 100:8, 100:16` = 100 steps at grid 4, refine to 8 for 100,
   refine to 16 for 100).
2. **Fixed baseline** — trains at the *final* grid size for the whole budget.

If refinement is pulling its weight, the adaptive run should match or beat the
fixed baseline's compliance in the same or less wall-clock time (its early
phases run on a much smaller parameter vector), and its loss curve should show
no lasting jump at the refinement boundaries (dashed lines) — proof the refit
carried the learned design across.


In [ ]:
# --- KAN Grid Refinement Controls ---
from models import train_lbfgs_adaptive_kan
refine_style = {"description_width": "initial"}

refine_category = widgets.Dropdown(
    options=CATEGORIES, value=CATEGORIES[0], description="Category:", style=refine_style,
)
refine_config = widgets.Dropdown(
    options=_config_options(refine_category.value), description="Config:", style=refine_style,
)

def _on_refine_category_change(change):
    refine_config.options = _config_options(change["new"])

refine_category.observe(_on_refine_category_change, names="value")

if "mbb_beam_96x32_0.5" in problems.PROBLEMS_BY_NAME:
    refine_category.value = "mbb_beam"
    refine_config.options = _config_options("mbb_beam")
    refine_config.value = "mbb_beam_96x32_0.5"

refine_kan_layers_text = widgets.Text(value="16, 16", description="Hidden layers:", style=refine_style)
refine_k = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Spline order (k):", style=refine_style)
refine_schedule_text = widgets.Text(
    value="100:4, 100:8, 100:16",
    description="Schedule (steps:grid, ...):",
    style=refine_style, layout=widgets.Layout(width="420px"),
)

refine_run_button = widgets.Button(description="Run Refinement Comparison", button_style="success", icon="play")
refine_output = widgets.Output()

_refine_run_lock = threading.Lock()


def _refine_best(ds):
    losses = ds.loss.values
    best_step = int(np.nanargmin(losses))
    compliance = float(losses[~np.isnan(losses)].min())
    design = np.clip(ds.design.isel(step=best_step).values, 0.0, 1.0)
    return compliance, design, losses


def run_grid_refinement(_button):
    if not _refine_run_lock.acquire(blocking=False):
        return
    refine_run_button.disabled = True
    try:
      with refine_output:
        clear_output(wait=True)
        try:
            p_name = refine_config.value
            problem = problems.PROBLEMS_BY_NAME[p_name]
            args = topo_api.specified_task(problem)
            layers = tuple(int(s.strip()) for s in refine_kan_layers_text.value.split(",") if s.strip())

            schedule = []
            for part in refine_schedule_text.value.split(","):
                steps_str, grid_str = part.strip().split(":")
                schedule.append((int(steps_str), int(grid_str)))
            total_steps = sum(s for s, _ in schedule)
            final_grid = schedule[-1][1]

            print("Problem: {} ({} x {}, density={})".format(
                p_name, problem.width, problem.height, problem.density))
            print("KAN: layers={}, k={}".format(list(layers), refine_k.value))
            print("Schedule: {}  (total {} steps, final grid {})".format(
                ", ".join("{} steps @ grid {}".format(s, g) for s, g in schedule),
                total_steps, final_grid))
            print()

            # 1. Adaptive: start at the first phase's grid, refine per schedule.
            print("[1/2] Adaptive coarse-to-fine run...")
            t0 = time.time()
            model_a = BaseKANModel(seed=0, args=args, kan_layers=layers,
                                    grid=schedule[0][1], k=refine_k.value)
            ds_a = train_lbfgs_adaptive_kan(model_a, schedule)
            time_a = time.time() - t0
            comp_a, design_a, losses_a = _refine_best(ds_a)
            print("  done in {:.2f}s, best compliance={:.4f}".format(time_a, comp_a))

            # 2. Fixed baseline: final grid from step one, same total budget.
            print("[2/2] Fixed-grid baseline (grid={} for all {} steps)...".format(final_grid, total_steps))
            t0 = time.time()
            model_f = BaseKANModel(seed=0, args=args, kan_layers=layers,
                                    grid=final_grid, k=refine_k.value)
            ds_f = train_lbfgs(model_f, total_steps, progress_every=PROGRESS_EVERY)
            time_f = time.time() - t0
            comp_f, design_f, losses_f = _refine_best(ds_f)
            print("  done in {:.2f}s, best compliance={:.4f}".format(time_f, comp_f))

            print()
            print("=" * 62)
            print("{:<34} {:>12} {:>12}".format("Condition", "Compliance", "Time"))
            print("-" * 62)
            print("{:<34} {:>12.4f} {:>10.2f}s".format(
                "Adaptive (grid {} -> {})".format(schedule[0][1], final_grid), comp_a, time_a))
            print("{:<34} {:>12.4f} {:>10.2f}s".format(
                "Fixed grid {}".format(final_grid), comp_f, time_f))
            print("=" * 62)
            better = comp_a <= comp_f
            print()
            print("Adaptive is {} by {:.2f} compliance ({:+.1f}% time)".format(
                "BETTER" if better else "worse", abs(comp_f - comp_a),
                100.0 * (time_a - time_f) / time_f))

            fig, axes = plt.subplots(1, 3, figsize=(16, 4))
            ax_a, ax_f, ax_curve = axes

            ax_a.imshow(1.0 - design_a, cmap="gray", vmin=0.0, vmax=1.0)
            ax_a.set_title("Adaptive\ncompliance={:.4f}, {:.1f}s".format(comp_a, time_a))
            ax_a.axis("off")

            ax_f.imshow(1.0 - design_f, cmap="gray", vmin=0.0, vmax=1.0)
            ax_f.set_title("Fixed grid {}\ncompliance={:.4f}, {:.1f}s".format(final_grid, comp_f, time_f))
            ax_f.axis("off")

            ax_curve.plot(np.arange(len(losses_a)), losses_a, label="adaptive")
            ax_curve.plot(np.arange(len(losses_f)), losses_f, label="fixed grid {}".format(final_grid))
            boundary = 0
            for steps, grid in schedule[:-1]:
                boundary += steps
                ax_curve.axvline(boundary, color="red", linestyle="--", linewidth=1, alpha=0.7)
            ax_curve.axvline(np.nan, color="red", linestyle="--", linewidth=1,
                              alpha=0.7, label="grid refinement")
            ax_curve.set_title("Compliance vs. step (refinements at dashed lines)")
            ax_curve.set_xlabel("Step")
            ax_curve.set_ylabel("Compliance")
            ax_curve.set_yscale("log")
            ax_curve.legend(fontsize=9)
            ax_curve.grid(True)

            plt.tight_layout()
            display(fig)
            plt.close(fig)
        except Exception:
            traceback.print_exc()
    finally:
        refine_run_button.disabled = False
        _refine_run_lock.release()


refine_run_button.on_click(run_grid_refinement)

display(widgets.VBox([
    widgets.HTML("<h3>Problem</h3>"),
    refine_category, refine_config,
    widgets.HTML("<hr><h3>Base KAN architecture</h3>"),
    refine_kan_layers_text, refine_k,
    widgets.HTML("<hr><h3>Refinement schedule</h3>"),
    refine_schedule_text,
    refine_run_button,
]))
display(refine_output)

Output()